# IA para Jugar Friday Night Funkin'
Este notebook cubre los pasos básicos para crear una IA que juegue *Friday Night Funkin'*. Incluye la configuración del entorno, captura de datos, preprocesamiento de imágenes, definición de la red neuronal, entrenamiento del modelo e integración básica.

In [ ]:
# Paso 1: Configuración del Entorno
!pip install numpy pandas matplotlib torch torchvision mss opencv-python

In [ ]:
# Paso 2: Captura de Datos del Juego
import mss
import numpy as np
import cv2
import matplotlib.pyplot as plt

with mss.mss() as sct:
    monitor = sct.monitors[0]
    img = np.array(sct.grab(monitor))
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

# Mostrar la imagen utilizando matplotlib
plt.imshow(img)
plt.axis('off')
plt.show()

In [ ]:
# Paso 3: Preprocesamiento de Imágenes
from PIL import Image
import torchvision.transforms as transforms

# Convertir la imagen de numpy array a PIL Image
img_pil = Image.fromarray(img)

# Definir las transformaciones
transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
])

# Aplicar las transformaciones
img_tensor = transform(img_pil)
print(img_tensor.shape)

In [ ]:
# Paso 4: Diseño de la Red Neuronal
import torch
import torch.nn as nn
import torch.nn.functional as F

class Net(nn.Module):
    def __init__(self):
        super(Net, self).__init__()
        self.conv1 = nn.Conv2d(3, 32, kernel_size=3, stride=1, padding=1)
        self.conv2 = nn.Conv2d(32, 64, kernel_size=3, stride=1, padding=1)
        self.fc1 = nn.Linear(64*56*56, 256)
        self.fc2 = nn.Linear(256, 4)  # Suponiendo que hay 4 acciones posibles

    def forward(self, x):
        x = F.relu(self.conv1(x))
        x = F.max_pool2d(x, 2)
        x = F.relu(self.conv2(x))
        x = F.max_pool2d(x, 2)
        x = x.view(-1, 64*56*56)
        x = F.relu(self.fc1(x))
        x = self.fc2(x)
        return x

net = Net()
print(net)

In [ ]:
# Paso 5: Entrenamiento del Modelo
import torch.optim as optim

# Definir criterio y optimizador
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(net.parameters(), lr=0.001)

# Datos de ejemplo (usa tu propio DataLoader)
# Debes reemplazar esto con tu propio DataLoader con datos de entrenamiento reales
trainloader = [(img_tensor, torch.tensor([1]))]  # Etiqueta ficticia

# Entrenamiento
for epoch in range(10):  # Número de épocas de entrenamiento
    running_loss = 0.0
    for i, data in enumerate(trainloader, 0):
        inputs, labels = data
        optimizer.zero_grad()
        outputs = net(inputs.unsqueeze(0))  # Agregar dimensión batch
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        running_loss += loss.item()
        if i % 100 == 99:  # Imprimir cada 100 mini-batches
            print(f'[Epoch {epoch + 1}, Batch {i + 1}] loss: {running_loss / 100:.3f}')
            running_loss = 0.0

In [ ]:
# Paso 6: Evaluación y Ajuste del Modelo
# Aquí puedes evaluar tu modelo con datos de prueba y ajustar los hiperparámetros según sea necesario.

In [ ]:
# Paso 7: Integración con el Juego
import pyautogui

# Ejemplo básico de cómo podría integrarse
def play_game(model, transform):
    with mss.mss() as sct:
        while True:
            monitor = sct.monitors[0]
            img = np.array(sct.grab(monitor))
            img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
            img_pil = Image.fromarray(img)
            img_tensor = transform(img_pil).unsqueeze(0)
            output = model(img_tensor)
            _, predicted = torch.max(output, 1)
            action = predicted.item()
            print(f'Predicted action: {action}')  # Añadido para depuración
            if action == 0:
                pyautogui.press('up')
            elif action == 1:
                pyautogui.press('down')
            elif action == 2:
                pyautogui.press('left')
            elif action == 3:
                pyautogui.press('right')

# Definir las transformaciones (asegúrate de definir el transform como en el paso 3)
transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
])

# Llamar a la función para empezar a jugar
# Nota: Esto es solo un ejemplo; necesitarás ajustar y expandir según tu juego
play_game(net, transform)